# LexFind — Kaggle GPU Embedder (Memory-Optimized)

**Step 2b of the Qdrant Ingestion Pipeline**

This notebook:
1. Installs `sentence-transformers`
2. Loads chunks from `/kaggle/input/` JSONL files **one file at a time** (memory-safe)
3. Embeds all chunks with `all-mpnet-base-v2` on T4 GPU (batch=128)
4. Saves `embeddings.npy` + `chunk_ids.json`

**Upload Instructions:**
- Upload your `.jsonl` file(s) as a Kaggle Dataset
- Add it to this notebook via: Notebook -> Data -> Add Dataset
- The files will appear at `/kaggle/input/<dataset-name>/`

**Hardware:** GPU T4 x2 | Accelerator: GPU

In [ ]:
# Cell 1 - Install dependencies
!pip install -q sentence-transformers==4.1.0

In [ ]:
# Cell 2 - Imports and GPU check
import json
import os
import gc
import glob
import numpy as np
import torch
from pathlib import Path

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'GPU count       : {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} - {props.total_memory / 1024**3:.1f} GB')

# Check available RAM
import psutil
ram = psutil.virtual_memory()
print(f'\nSystem RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available')

In [ ]:
# Cell 3 - Find input JSONL files
INPUT_GLOB = '/kaggle/input/**/*.jsonl'

jsonl_files = sorted(glob.glob(INPUT_GLOB, recursive=True))
print(f'Found {len(jsonl_files)} JSONL file(s):')
total_size = 0
for f in jsonl_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    total_size += size_mb
    print(f'  {f}  ({size_mb:.1f} MB)')
print(f'\nTotal input size: {total_size:.1f} MB')

if not jsonl_files:
    raise FileNotFoundError(
        'No .jsonl files found! Upload your chunks_for_embedding_*.jsonl '
        'as a Kaggle Dataset and add it to this notebook.'
    )

In [ ]:
# Cell 4 - Load model FIRST (before loading data, to ensure enough RAM)
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
BATCH_SIZE = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading model: {MODEL_NAME}')
print(f'Device: {DEVICE}')

model = SentenceTransformer(MODEL_NAME, device=DEVICE)

print(f'Max sequence length: {model.max_seq_length}')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')
print(f'\nModel loaded successfully!')

ram = psutil.virtual_memory()
print(f'RAM after model load: {ram.available / 1024**3:.1f} GB available')

In [ ]:
# Cell 5 - MEMORY-SAFE: Process ONE JSONL file at a time
# Instead of loading all 1.16M chunks into RAM at once (which caused OOM),
# we process each JSONL file independently, embed it, save a shard .npy,
# then free the memory before loading the next file.

OUTPUT_DIR = Path('/kaggle/working')
all_chunk_ids = []
shard_files = []
total_embedded = 0

for file_idx, jsonl_file in enumerate(jsonl_files):
    print(f'\n{"="*60}')
    print(f'Processing file {file_idx + 1}/{len(jsonl_files)}: {os.path.basename(jsonl_file)}')
    print(f'{"="*60}')
    
    # Load only texts and IDs from this file (not full JSON objects)
    texts = []
    chunk_ids = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                obj = json.loads(line)
                texts.append(obj['chunk_text'])
                chunk_ids.append(obj['chunk_id'])
                del obj  # Free JSON object immediately
    
    print(f'  Chunks in this file: {len(texts):,}')
    
    # Embed this file's chunks
    print(f'  Embedding with batch_size={BATCH_SIZE} ...')
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=DEVICE,
    )
    
    print(f'  Shape: {embeddings.shape}, Dtype: {embeddings.dtype}')
    
    # Save this shard
    shard_path = OUTPUT_DIR / f'embeddings_shard_{file_idx:03d}.npy'
    np.save(str(shard_path), embeddings)
    shard_files.append(shard_path)
    all_chunk_ids.extend(chunk_ids)
    total_embedded += len(texts)
    
    shard_mb = shard_path.stat().st_size / 1024 / 1024
    print(f'  Saved shard: {shard_path.name} ({shard_mb:.1f} MB)')
    
    # FREE MEMORY before next file
    del texts, chunk_ids, embeddings
    gc.collect()
    torch.cuda.empty_cache()
    
    ram = psutil.virtual_memory()
    print(f'  RAM available: {ram.available / 1024**3:.1f} GB')

print(f'\n{"="*60}')
print(f'All files processed! Total chunks embedded: {total_embedded:,}')
print(f'Shard files created: {len(shard_files)}')
print(f'{"="*60}')

In [ ]:
# Cell 6 - Merge all shards into final embeddings.npy + chunk_ids.json
# Free the model first to make room for concatenation
del model
gc.collect()
torch.cuda.empty_cache()

print('Merging embedding shards...')

# Load and concatenate shards one by one
all_embeddings = []
for shard_path in shard_files:
    print(f'  Loading {shard_path.name} ...')
    shard = np.load(str(shard_path))
    all_embeddings.append(shard)

embeddings = np.concatenate(all_embeddings, axis=0)
del all_embeddings
gc.collect()

print(f'\nFinal embeddings shape: {embeddings.shape}')
print(f'Final embeddings dtype: {embeddings.dtype}')
print(f'Total chunk_ids: {len(all_chunk_ids):,}')

assert embeddings.shape[0] == len(all_chunk_ids), (
    f'MISMATCH: {embeddings.shape[0]} embeddings vs {len(all_chunk_ids)} chunk_ids!'
)
assert embeddings.shape[1] == 768, f'Expected 768 dims, got {embeddings.shape[1]}'

# Save final outputs
EMB_PATH = OUTPUT_DIR / 'embeddings.npy'
IDS_PATH = OUTPUT_DIR / 'chunk_ids.json'

np.save(str(EMB_PATH), embeddings)
with open(IDS_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_chunk_ids, f)

emb_size_mb = EMB_PATH.stat().st_size / 1024 / 1024
ids_size_mb = IDS_PATH.stat().st_size / 1024 / 1024

print(f'\nSaved embeddings.npy  -> {EMB_PATH}  ({emb_size_mb:.1f} MB)')
print(f'Saved chunk_ids.json  -> {IDS_PATH}  ({ids_size_mb:.1f} MB)')

# Clean up shard files
for shard_path in shard_files:
    shard_path.unlink()
    print(f'  Deleted shard: {shard_path.name}')

print(f'\nDownload both files from the Kaggle Output tab.')
print(f'Place them in: D:\\LexFind\\backend\\scripts\\qdrant_ingestion\\')
print(f'Then run: python scripts/qdrant_ingestion/qdrant_ingestor.py')

In [ ]:
# Cell 7 - Verify embeddings are normalized (cosine-ready)
sample_norms = np.linalg.norm(embeddings[:100], axis=1)
print(f'Norms (first 100 vectors):')
print(f'  Min  : {sample_norms.min():.6f}')
print(f'  Max  : {sample_norms.max():.6f}')
print(f'  Mean : {sample_norms.mean():.6f}')
print()
print('All norms should be ~1.0 (cosine similarity ready) ✓' 
      if sample_norms.min() > 0.99 
      else 'WARNING: norms are not 1.0 - check normalize_embeddings=True')

print(f'\n--- SUMMARY ---')
print(f'Total vectors  : {embeddings.shape[0]:,}')
print(f'Dimensions     : {embeddings.shape[1]}')
print(f'Model          : all-mpnet-base-v2')
print(f'Normalized     : Yes (cosine-ready)')
print(f'\nReady for Qdrant ingestion!')